<a href="https://colab.research.google.com/github/BrajanNieto/DeepLearning/blob/main/Lab4_ComfyUI_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 4 - ComfyUI con InstantID

## Pipeline para Transformación de Identidad

Este notebook configura ComfyUI con:
- **InstantID**: Preservación de identidad facial
- **IP-Adapter**: Transferencia de estilo
- **ControlNet**: Control de pose
- **FaceDetailer**: Mejora de rostros


**Authors:**

Lopez Medina, Sebastian  
[sebastian.lopez@utec.edu.pe](mailto:sebastian.lopez@utec.edu.pe)

Nieto Espinoza, Brajan  
[brajan.nieto@utec.edu.pe](mailto:brajan.nieto@utec.edu.pe)

Tapia Chasquibol, Mateo  
[mateo.tapia@utec.edu.pe](mailto:mateo.tapia@utec.edu.pe)

---


## 1 Verificar GPU

Primero verifica que tienes GPU asignada.

In [ ]:
# Verificar GPU disponible
!nvidia-smi

import torch
print(f"\n PyTorch version: {torch.__version__}")
print(f" CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" GPU: {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Sun Apr 12 21:21:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2 Instalar Dependencias del Sistema

In [ ]:
import subprocess

print("Instalando dependencias del sistema...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq",
                "wget", "aria2", "libgl1-mesa-glx", "libglib2.0-0",
                "libsm6", "libxrender1", "libxext6"], check=True)
print("Dependencias del sistema instaladas")

Instalando dependencias del sistema...
Dependencias del sistema instaladas


## 3 Clonar ComfyUI e Instalar Requirements

In [ ]:
import os, subprocess, sys

COMFYUI_DIR = "/content/ComfyUI"

if not os.path.isdir(COMFYUI_DIR):
    print("Clonando ComfyUI...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/comfyanonymous/ComfyUI",
                    COMFYUI_DIR], check=True)
    print("ComfyUI clonado")
else:
    print("Actualizando ComfyUI...")
    subprocess.run(["git", "-C", COMFYUI_DIR, "pull"], check=True)
    print("ComfyUI actualizado")

print("\n Instalando requirements de ComfyUI...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "xformers!=0.0.18", "-r", f"{COMFYUI_DIR}/requirements.txt",
     "--extra-index-url", "https://download.pytorch.org/whl/cu121"],
    check=True
)
print("Requirements de ComfyUI instalados")

Actualizando ComfyUI...
ComfyUI actualizado

 Instalando requirements de ComfyUI...
Requirements de ComfyUI instalados


## 4 Instalar InsightFace (Requerido para InstantID)

In [ ]:
import subprocess, sys

print("Instalando InsightFace y dependencias...")

packages = [
    "insightface",
    "onnxruntime-gpu",
    "onnx",
    "albumentations",
    "opencv-python-headless"
]

for pkg in packages:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           capture_output=True, text=True)
    if result.returncode == 0:
        print(f"{pkg}")
    else:
        print(f"{pkg} - puede haber advertencias")

print("\n InsightFace y dependencias instaladas")

Instalando InsightFace y dependencias...
insightface
onnxruntime-gpu
onnx
albumentations
opencv-python-headless

 InsightFace y dependencias instaladas


## 5. Instalar Custom Nodes

Estos son los nodos personalizados necesarios para el pipeline.

In [ ]:
import os, subprocess, sys

CUSTOM_NODES_DIR = "/content/ComfyUI/custom_nodes"
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

# Custom nodes necesarios
CUSTOM_NODES = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager",
    "ComfyUI_InstantID": "https://github.com/cubiq/ComfyUI_InstantID",
    "ComfyUI_IPAdapter_plus": "https://github.com/cubiq/ComfyUI_IPAdapter_plus",
    "ComfyUI-Impact-Pack": "https://github.com/ltdrdata/ComfyUI-Impact-Pack",
    "ComfyUI_essentials": "https://github.com/cubiq/ComfyUI_essentials",
    "comfyui_controlnet_aux": "https://github.com/Fannovel16/comfyui_controlnet_aux",
}

print("Instalando Custom Nodes...\n")

for name, url in CUSTOM_NODES.items():
    node_path = os.path.join(CUSTOM_NODES_DIR, name)
    if not os.path.isdir(node_path):
        print(f"  Clonando {name}...")
        result = subprocess.run(["git", "clone", "--depth", "1", url, node_path],
                               capture_output=True, text=True)
        if result.returncode == 0:
            print(f"  {name}")
        else:
            print(f"  Error en {name}")
    else:
        print(f"  {name} (ya existe)")

print("\n Instalando requirements de custom nodes...")

for name in CUSTOM_NODES.keys():
    req_file = os.path.join(CUSTOM_NODES_DIR, name, "requirements.txt")
    if os.path.isfile(req_file):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file],
                      capture_output=True)

print("\n Todos los Custom Nodes instalados")

Instalando Custom Nodes...

  ComfyUI-Manager (ya existe)
  ComfyUI_InstantID (ya existe)
  ComfyUI_IPAdapter_plus (ya existe)
  ComfyUI-Impact-Pack (ya existe)
  ComfyUI_essentials (ya existe)
  comfyui_controlnet_aux (ya existe)

 Instalando requirements de custom nodes...

 Todos los Custom Nodes instalados


## 6 Crear Estructura de Carpetas para Modelos

In [ ]:
import os

MODELS_BASE = "/content/ComfyUI/models"

folders = [
    "checkpoints",
    "clip_vision",
    "controlnet",
    "instantid",
    "ipadapter",
    "insightface/models/antelopev2",
    "loras",
    "ultralytics/bbox",
    "ultralytics/segm",
    "vae",
    "sams"
]

print(" Creando estructura de carpetas...\n")
for folder in folders:
    path = os.path.join(MODELS_BASE, folder)
    os.makedirs(path, exist_ok=True)
    print(f"  {folder}")

print("\n Estructura de carpetas creada")

 Creando estructura de carpetas...

  checkpoints
  clip_vision
  controlnet
  instantid
  ipadapter
  insightface/models/antelopev2
  loras
  ultralytics/bbox
  ultralytics/segm
  vae
  sams

 Estructura de carpetas creada


## 7 Descargar Checkpoint SDXL

In [ ]:
import subprocess, os

CHECKPOINTS_DIR = "/content/ComfyUI/models/checkpoints"

# RealVisXL - Mejor para fotos realistas
MODEL_NAME = "RealVisXL_V4.0.safetensors"
MODEL_URL = "https://huggingface.co/SG161222/RealVisXL_V4.0/resolve/main/RealVisXL_V4.0.safetensors"

filepath = os.path.join(CHECKPOINTS_DIR, MODEL_NAME)

if not os.path.isfile(filepath):
    print(f" Descargando {MODEL_NAME}...")
    print("   (Este archivo es grande, ~6.5GB, puede tomar varios minutos)\n")
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--console-log-level=warn", "--summary-interval=30",
        "-d", CHECKPOINTS_DIR,
        "-o", MODEL_NAME,
        MODEL_URL
    ], check=True)
    print(f"\n {MODEL_NAME} descargado")
else:
    print(f" {MODEL_NAME} ya existe")

 RealVisXL_V4.0.safetensors ya existe


## 8 Descargar Modelos InstantID

In [ ]:
import subprocess, os

print("Descargando modelos InstantID...\n")

# InstantID IP-Adapter
INSTANTID_DIR = "/content/ComfyUI/models/instantid"
INSTANTID_URL = "https://huggingface.co/InstantX/InstantID/resolve/main/ip-adapter.bin"
INSTANTID_FILE = os.path.join(INSTANTID_DIR, "ip-adapter.bin")

if not os.path.isfile(INSTANTID_FILE):
    print("  InstantID IP-Adapter...")
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--console-log-level=error",
        "-d", INSTANTID_DIR,
        "-o", "ip-adapter.bin",
        INSTANTID_URL
    ], check=True)
    print("  InstantID IP-Adapter")
else:
    print("  InstantID IP-Adapter (ya existe)")

# InstantID ControlNet
CONTROLNET_DIR = "/content/ComfyUI/models/controlnet"
CONTROLNET_URL = "https://huggingface.co/InstantX/InstantID/resolve/main/ControlNetModel/diffusion_pytorch_model.safetensors"
CONTROLNET_FILE = os.path.join(CONTROLNET_DIR, "instantid-controlnet.safetensors")

if not os.path.isfile(CONTROLNET_FILE):
    print("   InstantID ControlNet...")
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--console-log-level=error",
        "-d", CONTROLNET_DIR,
        "-o", "instantid-controlnet.safetensors",
        CONTROLNET_URL
    ], check=True)
    print("   InstantID ControlNet")
else:
    print("   InstantID ControlNet (ya existe)")

print("\n Modelos InstantID listos")

Descargando modelos InstantID...

  InstantID IP-Adapter (ya existe)
   InstantID ControlNet (ya existe)

 Modelos InstantID listos


## 9️ Descargar Modelo InsightFace (antelopev2)

**CRÍTICO**: Sin estos archivos, InstantID no funcionará.

In [ ]:
import subprocess, os

INSIGHTFACE_DIR = "/content/ComfyUI/models/insightface/models/antelopev2"

ANTELOPEV2_FILES = {
    "1k3d68.onnx": "https://huggingface.co/DIAMONIK7777/antelopev2/resolve/main/1k3d68.onnx",
    "2d106det.onnx": "https://huggingface.co/DIAMONIK7777/antelopev2/resolve/main/2d106det.onnx",
    "genderage.onnx": "https://huggingface.co/DIAMONIK7777/antelopev2/resolve/main/genderage.onnx",
    "glintr100.onnx": "https://huggingface.co/DIAMONIK7777/antelopev2/resolve/main/glintr100.onnx",
    "scrfd_10g_bnkps.onnx": "https://huggingface.co/DIAMONIK7777/antelopev2/resolve/main/scrfd_10g_bnkps.onnx"
}

print(" Descargando modelo antelopev2 para InsightFace...\n")

for filename, url in ANTELOPEV2_FILES.items():
    filepath = os.path.join(INSIGHTFACE_DIR, filename)
    if not os.path.isfile(filepath):
        print(f"   {filename}...")
        subprocess.run([
            "aria2c", "-x", "16", "-s", "16", "-k", "1M",
            "--console-log-level=error",
            "-d", INSIGHTFACE_DIR,
            "-o", filename,
            url
        ], check=True)
        print(f"   {filename}")
    else:
        print(f"   {filename} (ya existe)")

print("\n antelopev2 completo")

# Verificar
files = os.listdir(INSIGHTFACE_DIR)
print(f"\n Archivos en antelopev2: {len(files)} de 5 requeridos")

 Descargando modelo antelopev2 para InsightFace...

   1k3d68.onnx (ya existe)
   2d106det.onnx (ya existe)
   genderage.onnx (ya existe)
   glintr100.onnx (ya existe)
   scrfd_10g_bnkps.onnx (ya existe)

 antelopev2 completo

 Archivos en antelopev2: 5 de 5 requeridos


## 10 Descargar CLIP Vision

In [ ]:
import subprocess, os

CLIP_DIR = "/content/ComfyUI/models/clip_vision"
CLIP_URL = "https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/image_encoder/model.safetensors"
CLIP_FILE = os.path.join(CLIP_DIR, "CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors")

if not os.path.isfile(CLIP_FILE):
    print(" Descargando CLIP Vision para SDXL...")
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--console-log-level=error",
        "-d", CLIP_DIR,
        "-o", "CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
        CLIP_URL
    ], check=True)
    print(" CLIP Vision descargado")
else:
    print(" CLIP Vision ya existe")

 CLIP Vision ya existe


## 11 Descargar Modelo Face Detector (para FaceDetailer)




In [ ]:
import subprocess, os

BBOX_DIR = "/content/ComfyUI/models/ultralytics/bbox"
BBOX_URL = "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt"
BBOX_FILE = os.path.join(BBOX_DIR, "face_yolov8m.pt")

if not os.path.isfile(BBOX_FILE):
    print(" Descargando Face Detector...")
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--console-log-level=error",
        "-d", BBOX_DIR,
        "-o", "face_yolov8m.pt",
        BBOX_URL
    ], check=True)
    print(" Face Detector descargado")
else:
    print(" Face Detector ya existe")

 Face Detector ya existe


## 12 Verificar Instalación Completa

In [ ]:
import os

print(" Verificando instalación...\n")

checks = {
    "ComfyUI": "/content/ComfyUI/main.py",
    "InstantID Node": "/content/ComfyUI/custom_nodes/ComfyUI_InstantID",
    "IP-Adapter Node": "/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus",
    "Impact Pack": "/content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack",
    "Checkpoint SDXL": "/content/ComfyUI/models/checkpoints/RealVisXL_V4.0.safetensors",
    "InstantID Model": "/content/ComfyUI/models/instantid/ip-adapter.bin",
    "InstantID ControlNet": "/content/ComfyUI/models/controlnet/instantid-controlnet.safetensors",
    "antelopev2 (5 archivos)": "/content/ComfyUI/models/insightface/models/antelopev2",
    "CLIP Vision": "/content/ComfyUI/models/clip_vision/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
}

all_ok = True
for name, path in checks.items():
    if os.path.exists(path):
        if "antelopev2" in name:
            files = os.listdir(path)
            if len(files) >= 5:
                print(f"   {name}")
            else:
                print(f"   {name} - Solo {len(files)} archivos de 5")
                all_ok = False
        else:
            print(f"   {name}")
    else:
        print(f"   {name} - NO ENCONTRADO")
        all_ok = False

print()
if all_ok:
    print("listo! Puedes continuar a subir tus fotos.")
else:
    print(" Algunos componentes faltan. Revisa las celdas anteriores.")

 Verificando instalación...

   ComfyUI
   InstantID Node
   IP-Adapter Node
   Impact Pack
   Checkpoint SDXL
   InstantID Model
   InstantID ControlNet
   antelopev2 (5 archivos)
   CLIP Vision

listo! Puedes continuar a subir tus fotos.


## 13 Subir Fotos del Grupo


In [ ]:
import subprocess, os

# Configuración del repositorio
GITHUB_BASE = "https://raw.githubusercontent.com/BrajanNieto/DeepLearning/main/pictures"
INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

# Fotos de los integrantes
FOTOS = {
    "BNE.png": f"{GITHUB_BASE}/BNE.png",
    "MTC.png": f"{GITHUB_BASE}/MTC.png",
    "SLM.png": f"{GITHUB_BASE}/SLM.png",
    "Team.png": f"{GITHUB_BASE}/Team.png"
}

print(" Descargando fotos desde GitHub...\n")
print(f"   Repositorio: BrajanNieto/DeepLearning/pictures\n")

for filename, url in FOTOS.items():
    filepath = os.path.join(INPUT_DIR, filename)
    if not os.path.isfile(filepath):
        print(f"   {filename}...")
        result = subprocess.run(["wget", "-q", "-O", filepath, url])
        if result.returncode == 0 and os.path.isfile(filepath):
            size = os.path.getsize(filepath) / 1024
            print(f"   {filename} ({size:.1f} KB)")
        else:
            print(f"   Error descargando {filename}")
    else:
        print(f"   {filename} (ya existe)")

# Verificar
print("\n Fotos disponibles:")
for f in os.listdir(INPUT_DIR):
    if f.endswith('.png'):
        print(f"   • {f}")

print(f"\n Fotos listas en: {INPUT_DIR}")

 Descargando fotos desde GitHub...

   Repositorio: BrajanNieto/DeepLearning/pictures

   BNE.png...
   BNE.png (431.9 KB)
   MTC.png...
   MTC.png (394.2 KB)
   SLM.png...
   SLM.png (175.5 KB)
   Team.png...
   Team.png (1117.6 KB)

 Fotos disponibles:
   • SLM.png
   • MTC.png
   • BNE.png
   • Team.png
   • example.png

 Fotos listas en: /content/ComfyUI/input


## 14 Crear Carpetas de Output por Integrante

In [ ]:
import os

OUTPUT_BASE = "/content/ComfyUI/output"

# Estructura de carpetas para resultados
carpetas = [
    "individual/BNE/leve",
    "individual/BNE/moderado",
    "individual/BNE/fuerte",
    "individual/MTC/leve",
    "individual/MTC/moderado",
    "individual/MTC/fuerte",
    "individual/SLM/leve",
    "individual/SLM/moderado",
    "individual/SLM/fuerte",
    "grupal"
]

print(" Creando estructura de carpetas para resultados...\n")
for carpeta in carpetas:
    path = os.path.join(OUTPUT_BASE, carpeta)
    os.makedirs(path, exist_ok=True)
    print(f"   {carpeta}")

print("\n Estructura de output lista")

 Creando estructura de carpetas para resultados...

   individual/BNE/leve
   individual/BNE/moderado
   individual/BNE/fuerte
   individual/MTC/leve
   individual/MTC/moderado
   individual/MTC/fuerte
   individual/SLM/leve
   individual/SLM/moderado
   individual/SLM/fuerte
   grupal

 Estructura de output lista


## 15 Definir Prompts para Cada Nivel de Transformación

In [ ]:
# Configuración de prompts por nivel

PROMPTS = {
    # ========== NIVEL LEVE (Alta preservación: weight 0.85-1.0) ==========
    "leve": {
        "golden_hour": {
            "positive": "professional portrait photo, golden hour lighting, warm sunlight, outdoor background, soft shadows, high quality, 8k, sharp focus",
            "negative": "cartoon, anime, illustration, painting, drawing, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text",
            "weight": 0.90,
            "cfg": 4.5
        },
        "estudio": {
            "positive": "professional studio portrait, softbox lighting, neutral gray background, corporate headshot, sharp focus, professional attire, high quality photography",
            "negative": "cartoon, anime, illustration, painting, drawing, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text",
            "weight": 0.92,
            "cfg": 4.0
        },
        "cine_noir": {
            "positive": "cinematic noir portrait, dramatic side lighting, black and white, film grain, classic hollywood style, elegant, high contrast shadows",
            "negative": "cartoon, anime, colorful, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text",
            "weight": 0.88,
            "cfg": 5.0
        }
    },

    # ========== NIVEL MODERADO (Preservación media: weight 0.6-0.75) ==========
    "moderado": {
        "lab_futurista": {
            "positive": "scientist in futuristic laboratory, holographic displays, neon blue lighting, high-tech equipment, sci-fi aesthetic, wearing lab coat, detailed environment",
            "negative": "cartoon, anime, illustration, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text, extra limbs",
            "weight": 0.70,
            "cfg": 5.5
        },
        "cyberpunk": {
            "positive": "cyberpunk street scene, neon lights, rain, futuristic city, night atmosphere, blade runner aesthetic, cinematic, wearing tech jacket",
            "negative": "cartoon, anime, illustration, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text",
            "weight": 0.68,
            "cfg": 5.5
        },
        "editorial": {
            "positive": "high fashion editorial photo, dramatic studio lighting, vogue magazine style, artistic composition, professional model pose, elegant attire",
            "negative": "cartoon, anime, illustration, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text, casual clothes",
            "weight": 0.72,
            "cfg": 5.0
        }
    },

    # ========== NIVEL FUERTE (Transformación agresiva: weight 0.4-0.55) ==========
    "fuerte": {
        "cientifico_1920": {
            "positive": "1920s scientist portrait, vintage sepia tone, old laboratory, period accurate clothing, historical photograph style, aged paper texture, classic attire",
            "negative": "modern, cartoon, anime, colorful, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text",
            "weight": 0.52,
            "cfg": 6.0
        },
        "explorador_polar": {
            "positive": "antarctic explorer, heavy winter gear, snow storm background, expedition photo, vintage exploration photography, dramatic lighting, frost on face, fur hood",
            "negative": "cartoon, anime, illustration, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text, summer clothes",
            "weight": 0.50,
            "cfg": 6.5
        },
        "astronauta_luna": {
            "positive": "astronaut on moon surface, NASA spacesuit, earth in background, lunar landscape, realistic space photography, dramatic lighting, helmet visor reflection",
            "negative": "cartoon, anime, illustration, bad anatomy, deformed face, ugly, blurry, low quality, watermark, text",
            "weight": 0.48,
            "cfg": 6.5
        }
    }
}

# Prompts para escenas grupales
PROMPTS_GRUPALES = {
    "equipo_ia": {
        "positive": "team of AI researchers, modern tech office, multiple monitors with code, collaborative workspace, professional group photo, natural lighting, standing together",
        "negative": "cartoon, anime, illustration, bad anatomy, deformed faces, ugly, blurry, low quality, watermark, text",
        "weight": 0.65,
        "cfg": 4.5
    },
    "conferencia": {
        "positive": "conference panel speakers, professional stage, microphones, presentation screen behind, business formal attire, professional lighting, three speakers",
        "negative": "cartoon, anime, illustration, bad anatomy, deformed faces, ugly, blurry, low quality, watermark, text",
        "weight": 0.62,
        "cfg": 4.5
    },
    "marte": {
        "positive": "mars exploration team, red planet surface, space suits, NASA mission style, dramatic martian landscape, team photo, orange rocky terrain",
        "negative": "cartoon, anime, illustration, bad anatomy, deformed faces, ugly, blurry, low quality, watermark, text",
        "weight": 0.55,
        "cfg": 5.5
    }
}

print("✅ Prompts definidos:")
print(f"   • Nivel leve: {len(PROMPTS['leve'])} escenarios")
print(f"   • Nivel moderado: {len(PROMPTS['moderado'])} escenarios")
print(f"   • Nivel fuerte: {len(PROMPTS['fuerte'])} escenarios")
print(f"   • Grupales: {len(PROMPTS_GRUPALES)} escenarios")

✅ Prompts definidos:
   • Nivel leve: 3 escenarios
   • Nivel moderado: 3 escenarios
   • Nivel fuerte: 3 escenarios
   • Grupales: 3 escenarios


## 16 Iniciar ComfyUI

Después de ejecutar esta celda:
1. Abre la URL que aparece
2. Arrastra el archivo `workflow_instantid.json` al canvas
3. Modifica la imagen de entrada y los prompts según la tabla de la celda 15

In [ ]:
from google.colab.output import eval_js
import subprocess, threading, time

COMFYUI_DIR = "/content/ComfyUI"

print("Iniciando ComfyUI...")
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd=COMFYUI_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

ready = threading.Event()

def log():
    for line in comfy_proc.stdout:
        print("[ComfyUI]", line.rstrip())
        if "To see the GUI" in line:
            ready.set()

threading.Thread(target=log, daemon=True).start()
ready.wait(timeout=120)
time.sleep(3)

print("\n" + "="*60)
print("LISTO - Usa el proxy de Colab")
print("="*60)
print(eval_js("google.colab.kernel.proxyPort(8188)"))
print("="*60)

comfy_proc.wait()

Iniciando ComfyUI...
[ComfyUI] [ComfyUI-Manager] Using `uv` as Python module for pip operations.
[ComfyUI] Using Python 3.12.13 environment at: /usr
[ComfyUI] [START] Security scan
[ComfyUI] [DONE] Security scan
[ComfyUI] ## ComfyUI-Manager: installing dependencies done.
[ComfyUI] ** ComfyUI startup time: 2026-04-12 22:17:57.547
[ComfyUI] ** Platform: Linux
[ComfyUI] ** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[ComfyUI] ** Python executable: /usr/bin/python3
[ComfyUI] ** ComfyUI Path: /content/ComfyUI
[ComfyUI] ** ComfyUI Base Folder Path: /content/ComfyUI
[ComfyUI] ** User directory: /content/ComfyUI/user
[ComfyUI] ** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
[ComfyUI] ** Log path: /content/ComfyUI/user/comfyui.log
[ComfyUI] Using Python 3.12.13 environment at: /usr
[ComfyUI] Using Python 3.12.13 environment at: /usr
[ComfyUI] 
[ComfyUI] Prestartup times for custom nodes:
[ComfyUI]    1.4 seconds: /content/ComfyUI/custom_nodes/Co

KeyboardInterrupt: 

---
##  Descargar Resultados


In [ ]:
import subprocess
import threading
import time
import json
import urllib.request
import os
import shutil
from google.colab import files

COMFYUI_DIR = "/content/ComfyUI"
COMFYUI_URL = "http://127.0.0.1:8188"

# 1. Iniciar ComfyUI en background
print("Iniciando ComfyUI...")
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd=COMFYUI_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

ready = threading.Event()

def log():
    for line in comfy_proc.stdout:
        if "To see the GUI" in line or "Starting server" in line:
            ready.set()

threading.Thread(target=log, daemon=True).start()
ready.wait(timeout=120)
time.sleep(5)
print("ComfyUI listo.\n")

# 2. Funcion para generar imagen
def generar(foto, prompt, weight, cfg, nombre):
    workflow = {
        "1": {"class_type": "LoadImage", "inputs": {"image": foto}},
        "2": {"class_type": "InstantIDModelLoader", "inputs": {"instantid_file": "ip-adapter.bin"}},
        "3": {"class_type": "InstantIDFaceAnalysis", "inputs": {"provider": "CUDA"}},
        "4": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": "RealVisXL_V4.0.safetensors"}},
        "5": {"class_type": "ControlNetLoader", "inputs": {"control_net_name": "instantid-controlnet.safetensors"}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["4", 1]}},
        "7": {"class_type": "CLIPTextEncode", "inputs": {"text": "cartoon, anime, illustration, bad anatomy, deformed face, ugly, blurry, low quality", "clip": ["4", 1]}},
        "8": {"class_type": "ApplyInstantID", "inputs": {"instantid": ["2", 0], "insightface": ["3", 0], "control_net": ["5", 0], "image": ["1", 0], "model": ["4", 0], "positive": ["6", 0], "negative": ["7", 0], "weight": weight, "start_at": 0.0, "end_at": 1.0}},
        "9": {"class_type": "EmptyLatentImage", "inputs": {"width": 1024, "height": 1024, "batch_size": 1}},
        "10": {"class_type": "KSampler", "inputs": {"model": ["8", 0], "positive": ["8", 1], "negative": ["8", 2], "latent_image": ["9", 0], "seed": 42, "steps": 25, "cfg": cfg, "sampler_name": "euler", "scheduler": "normal", "denoise": 1.0}},
        "11": {"class_type": "VAEDecode", "inputs": {"samples": ["10", 0], "vae": ["4", 2]}},
        "12": {"class_type": "SaveImage", "inputs": {"images": ["11", 0], "filename_prefix": nombre}}
    }

    data = json.dumps({"prompt": workflow}).encode('utf-8')
    req = urllib.request.Request(f"{COMFYUI_URL}/prompt", data=data, headers={'Content-Type': 'application/json'})
    try:
        urllib.request.urlopen(req, timeout=10)
        return True
    except:
        return False

# 3. Esperar a que una imagen termine
def esperar_cola():
    while True:
        try:
            req = urllib.request.Request(f"{COMFYUI_URL}/queue")
            resp = urllib.request.urlopen(req, timeout=5)
            data = json.loads(resp.read())
            if len(data.get("queue_running", [])) == 0 and len(data.get("queue_pending", [])) == 0:
                return
        except:
            pass
        time.sleep(3)

# 4. Configuraciones
CONFIGS = [
    # BNE
    ("BNE.png", "professional portrait, golden hour lighting, warm sunlight, high quality", 0.90, 4.5, "BNE_leve_golden"),
    ("BNE.png", "professional studio portrait, softbox lighting, gray background", 0.92, 4.0, "BNE_leve_estudio"),
    ("BNE.png", "cinematic noir portrait, dramatic side lighting, black and white", 0.88, 5.0, "BNE_leve_noir"),
    ("BNE.png", "scientist in futuristic lab, holographic displays, neon blue", 0.70, 5.5, "BNE_mod_lab"),
    ("BNE.png", "cyberpunk street, neon lights, rain, blade runner aesthetic", 0.68, 5.5, "BNE_mod_cyber"),
    ("BNE.png", "1920s scientist, vintage sepia, old laboratory", 0.52, 6.0, "BNE_fuerte_1920"),
    ("BNE.png", "astronaut on moon, NASA spacesuit, earth background", 0.48, 6.5, "BNE_fuerte_luna"),

    # MTC
    ("MTC.png", "professional portrait, golden hour lighting, warm sunlight, high quality", 0.90, 4.5, "MTC_leve_golden"),
    ("MTC.png", "professional studio portrait, softbox lighting, gray background", 0.92, 4.0, "MTC_leve_estudio"),
    ("MTC.png", "cinematic noir portrait, dramatic side lighting, black and white", 0.88, 5.0, "MTC_leve_noir"),
    ("MTC.png", "scientist in futuristic lab, holographic displays, neon blue", 0.70, 5.5, "MTC_mod_lab"),
    ("MTC.png", "cyberpunk street, neon lights, rain, blade runner aesthetic", 0.68, 5.5, "MTC_mod_cyber"),
    ("MTC.png", "1920s scientist, vintage sepia, old laboratory", 0.52, 6.0, "MTC_fuerte_1920"),
    ("MTC.png", "astronaut on moon, NASA spacesuit, earth background", 0.48, 6.5, "MTC_fuerte_luna"),

    # SLM
    ("SLM.png", "professional portrait, golden hour lighting, warm sunlight, high quality", 0.90, 4.5, "SLM_leve_golden"),
    ("SLM.png", "professional studio portrait, softbox lighting, gray background", 0.92, 4.0, "SLM_leve_estudio"),
    ("SLM.png", "cinematic noir portrait, dramatic side lighting, black and white", 0.88, 5.0, "SLM_leve_noir"),
    ("SLM.png", "scientist in futuristic lab, holographic displays, neon blue", 0.70, 5.5, "SLM_mod_lab"),
    ("SLM.png", "cyberpunk street, neon lights, rain, blade runner aesthetic", 0.68, 5.5, "SLM_mod_cyber"),
    ("SLM.png", "1920s scientist, vintage sepia, old laboratory", 0.52, 6.0, "SLM_fuerte_1920"),
    ("SLM.png", "astronaut on moon, NASA spacesuit, earth background", 0.48, 6.5, "SLM_fuerte_luna"),
]

# 5. Generar todas las imagenes
print(f"Generando {len(CONFIGS)} imagenes...")
print("="*50)

for i, (foto, prompt, weight, cfg, nombre) in enumerate(CONFIGS):
    print(f"[{i+1}/{len(CONFIGS)}] {nombre}...")
    if generar(foto, prompt, weight, cfg, nombre):
        esperar_cola()
        print(f"         Listo.")
    else:
        print(f"         Error.")
    time.sleep(2)

print("\n" + "="*50)
print("Generacion completada.")

# 6. Detener ComfyUI
comfy_proc.terminate()
time.sleep(2)

# 7. Descargar resultados
OUTPUT_DIR = "/content/ComfyUI/output"
total = sum(1 for r, d, f in os.walk(OUTPUT_DIR) for x in f if x.endswith('.png'))
print(f"\nImagenes generadas: {total}")

if total > 0:
    zip_path = "/content/Lab4_Resultados"
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    print("Descargando ZIP...")
    files.download(f"{zip_path}.zip")

Iniciando ComfyUI...
ComfyUI listo.

Generando 21 imagenes...
[1/21] BNE_leve_golden...
         Listo.
[2/21] BNE_leve_estudio...
         Listo.
[3/21] BNE_leve_noir...
         Listo.
[4/21] BNE_mod_lab...
         Listo.
[5/21] BNE_mod_cyber...
         Listo.
[6/21] BNE_fuerte_1920...
         Listo.
[7/21] BNE_fuerte_luna...
         Listo.
[8/21] MTC_leve_golden...
         Listo.
[9/21] MTC_leve_estudio...
         Listo.
[10/21] MTC_leve_noir...
         Listo.
[11/21] MTC_mod_lab...
         Listo.
[12/21] MTC_mod_cyber...
         Listo.
[13/21] MTC_fuerte_1920...
         Listo.
[14/21] MTC_fuerte_luna...
         Listo.
[15/21] SLM_leve_golden...
         Listo.
[16/21] SLM_leve_estudio...
         Listo.
[17/21] SLM_leve_noir...
         Listo.
[18/21] SLM_mod_lab...
         Listo.
[19/21] SLM_mod_cyber...
         Listo.
[20/21] SLM_fuerte_1920...
         Listo.
[21/21] SLM_fuerte_luna...
         Listo.

Generacion completada.

Imagenes generadas: 85
Descargando ZIP..

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>